# Unit-test notebook: Clickstream Sessionization
Set `SOLUTION` to the candidate file, run all cells. Each cell checks one rule from `task/instruction.md`.

In [ ]:
import importlib.util, json, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/'task').exists() else Path.cwd().parent
SOLUTION = os.environ.get('SOLUTION', str(ROOT/'solution'/'solution.py'))
spec = importlib.util.spec_from_file_location('candidate', SOLUTION)
cand = importlib.util.module_from_spec(spec); spec.loader.exec_module(cand)
report = cand.sessionize(str(ROOT/'task'/'data'/'events.csv'))
results = {}
def check(name, cond):
    results[name] = bool(cond); print(('PASS ' if cond else 'FAIL ') + name)

In [ ]:
check('exact 30-min gap continues', report.get('u_boundary') == {'sessions': 2, 'longest_session_s': 1800, 'events': 3})

In [ ]:
check('mixed offsets ordered by instant', report.get('u_offsets') == {'sessions': 1, 'longest_session_s': 2400, 'events': 3})

In [ ]:
check('logout ends session', report.get('u_logout') == {'sessions': 2, 'longest_session_s': 300, 'events': 4})

In [ ]:
check('dedup keeps earliest ingestion', report.get('u_dupes') == {'sessions': 1, 'longest_session_s': 1500, 'events': 2})

In [ ]:
check('all-invalid user omitted', 'u_invalid' not in report)

In [ ]:
check('invalid row ignored', report.get('u_mixed_invalid') == {'sessions': 2, 'longest_session_s': 0, 'events': 2})

In [ ]:
check('tie broken by event_id', report.get('u_tie') == {'sessions': 2, 'longest_session_s': 60, 'events': 3})

In [ ]:
check('plain int outputs', all(type(v) is int for s in report.values() for v in s.values()))

In [ ]:
exp = json.loads((ROOT/'tests'/'expected_background.json').read_text())
check('background users match reference', {u: report.get(u) for u in exp} == exp)

In [ ]:
print(f"{sum(results.values())}/{len(results)} passed")
print('failed:', [k for k, v in results.items() if not v])